# Sample Training Data for Quality Labeling

Samples ~10K documents per 25-year period from **both English corpus and additional news data**,
proportional to their document counts. No cleaning masks needed — GPT labeling handles quality gating.

**English:** `D:\hist_LLM\corpus\raw\{year}\subset_*.parquet` (col: `text`, `identifier`)  
**Additional:**
- NYT: `D:\hist_LLM\additional_data\raw\news_archives\NYT_filtered_500char\nyt_{year}.parquet` (col: `combined_text`, `_id`)
- Economist: `D:\hist_LLM\additional_data\raw\news_archives\Economist\economist_{year}-*.parquet` (col: `ocr_text`, `article_id`)
- FT: `D:\hist_LLM\additional_data\raw\news_archives\FT\{year}.parquet` (col: `text_cleaned`, `id`)
- Newswire: `D:\hist_LLM\additional_data\raw\newswire\{year}_data_clean.json` (col: `cleaned_article`)

**Output:** `D:\hist_LLM\processing\sample_data\training_samples_{period}.parquet`  
Schema: `[text, original_index, year, source]`

In [ ]:
import pandas as pd
import json
import pyarrow.parquet as pq
from pathlib import Path
import numpy as np
from tqdm.auto import tqdm

# --- CONFIG ---
ENGLISH_RAW_DIR = Path(r"D:\hist_LLM\corpus\raw")
ADDITIONAL_RAW_DIR = Path(r"D:\hist_LLM\additional_data\raw")
SAMPLE_OUTPUT_DIR = Path(r"D:\hist_LLM\processing\sample_data")
SAMPLE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SAMPLE_SIZE = 10_000
RANDOM_STATE = 42

# Additional data collection configs: (dir, file_pattern, text_col, id_col)
ADDITIONAL_COLLECTIONS = {
    "nyt": {
        "dir": ADDITIONAL_RAW_DIR / "news_archives" / "NYT_filtered_500char",
        "file_pattern": "nyt_{year}.parquet",
        "text_col": "combined_text",
        "id_col": "_id",
    },
    "economist": {
        "dir": ADDITIONAL_RAW_DIR / "news_archives" / "Economist",
        "file_pattern": "economist_{year}-*.parquet",  # multiple weekly files per year
        "text_col": "ocr_text",
        "id_col": "article_id",
    },
    "ft": {
        "dir": ADDITIONAL_RAW_DIR / "news_archives" / "FT",
        "file_pattern": "{year}.parquet",
        "text_col": "text_cleaned",
        "id_col": "id",
    },
    "newswire": {
        "dir": ADDITIONAL_RAW_DIR / "newswire",
        "file_pattern": "{year}_data_clean.json",
        "text_col": "cleaned_article",
        "id_col": None,  # index-based
    },
}


def count_english_docs(year: int) -> int:
    """Count docs in English raw data for a year (fast metadata read)."""
    year_dir = ENGLISH_RAW_DIR / str(year)
    if not year_dir.exists():
        return 0
    total = 0
    for f in year_dir.glob("subset_*.parquet"):
        try:
            total += pq.ParquetFile(f).metadata.num_rows
        except:
            pass
    return total


def count_additional_docs(year: int) -> dict:
    """Count docs per additional collection for a year."""
    counts = {}
    for name, cfg in ADDITIONAL_COLLECTIONS.items():
        if name == "economist":
            files = list(cfg["dir"].glob(f"economist_{year}-*.parquet"))
            total = sum(pq.ParquetFile(f).metadata.num_rows for f in files)
        elif name == "newswire":
            path = cfg["dir"] / f"{year}_data_clean.json"
            if path.exists() and year != 1957:  # 1957 is corrupted
                try:
                    with open(path, 'r', encoding='utf-8') as f:
                        data = json.load(f)
                    total = len(data)
                except:
                    total = 0
            else:
                total = 0
        else:
            pattern = cfg["file_pattern"].format(year=year)
            path = cfg["dir"] / pattern
            total = pq.ParquetFile(path).metadata.num_rows if path.exists() else 0
        if total > 0:
            counts[name] = total
    return counts


def sample_english(year: int, n: int, rng: np.random.Generator) -> list:
    """Sample n docs from English raw data for a year."""
    year_dir = ENGLISH_RAW_DIR / str(year)
    if not year_dir.exists():
        return []
    
    files = sorted(year_dir.glob("subset_*.parquet"))
    if not files:
        return []
    
    # Collect all rows with their file info
    all_rows = []
    for f in files:
        df = pd.read_parquet(f, columns=["identifier", "text"])
        for idx, row in df.iterrows():
            if row["text"] and str(row["text"]).strip():
                all_rows.append({
                    "text": row["text"],
                    "original_index": row["identifier"],
                    "year": year,
                    "source": "english",
                })
    
    if not all_rows:
        return []
    
    n = min(n, len(all_rows))
    indices = rng.choice(len(all_rows), size=n, replace=False)
    return [all_rows[i] for i in indices]


def sample_additional(year: int, collection: str, n: int, rng: np.random.Generator) -> list:
    """Sample n docs from an additional data collection for a year."""
    cfg = ADDITIONAL_COLLECTIONS[collection]
    
    if collection == "newswire":
        if year == 1957:
            return []
        path = cfg["dir"] / f"{year}_data_clean.json"
        if not path.exists():
            return []
        try:
            with open(path, 'r', encoding='utf-8') as f:
                data = json.load(f)
        except:
            return []
        n = min(n, len(data))
        indices = rng.choice(len(data), size=n, replace=False)
        return [{
            "text": data[i].get("cleaned_article", ""),
            "original_index": f"newswire_{year}_{i}",
            "year": year,
            "source": "newswire",
        } for i in indices if data[i].get("cleaned_article")]
    
    elif collection == "economist":
        files = sorted(cfg["dir"].glob(f"economist_{year}-*.parquet"))
        if not files:
            return []
        dfs = [pd.read_parquet(f, columns=[cfg["id_col"], cfg["text_col"]]) for f in files]
        df = pd.concat(dfs, ignore_index=True)
    else:
        pattern = cfg["file_pattern"].format(year=year)
        path = cfg["dir"] / pattern
        if not path.exists():
            return []
        df = pd.read_parquet(path, columns=[cfg["id_col"], cfg["text_col"]])
    
    # Filter empty text
    df = df.dropna(subset=[cfg["text_col"]])
    df = df[df[cfg["text_col"]].str.strip() != ""]
    
    if df.empty:
        return []
    
    n = min(n, len(df))
    sampled = df.sample(n=n, random_state=rng.integers(2**31))
    
    return [{
        "text": row[cfg["text_col"]],
        "original_index": str(row[cfg["id_col"]]),
        "year": year,
        "source": collection,
    } for _, row in sampled.iterrows()]


def collect_combined_samples(target_years: list):
    """Sample 10K docs from English + additional data, proportional to doc counts."""
    rng = np.random.default_rng(RANDOM_STATE)
    
    # Phase 1: Count docs per source per year
    print("Counting documents per source...")
    year_english = {}
    year_additional = {}  # {year: {collection: count}}
    
    for year in tqdm(target_years, desc="Counting"):
        year_english[year] = count_english_docs(year)
        year_additional[year] = count_additional_docs(year)
    
    total_english = sum(year_english.values())
    total_additional = sum(sum(c.values()) for c in year_additional.values())
    total_all = total_english + total_additional
    
    if total_all == 0:
        print("No documents found!")
        return
    
    # Calculate proportional split
    n_english = round(SAMPLE_SIZE * total_english / total_all)
    n_additional = SAMPLE_SIZE - n_english
    
    print(f"\nTotal docs: {total_all:,} (English: {total_english:,}, Additional: {total_additional:,})")
    print(f"Sample split: {n_english} English + {n_additional} additional = {SAMPLE_SIZE}")
    
    # Phase 2: Sample English (distribute across years proportionally)
    print("\nSampling English...")
    all_samples = []
    
    for year in tqdm(target_years, desc="English"):
        if year_english[year] == 0:
            continue
        year_n = max(1, round(n_english * year_english[year] / total_english))
        samples = sample_english(year, year_n, rng)
        all_samples.extend(samples)
    
    # Phase 3: Sample additional (distribute across collections proportionally)
    if n_additional > 0:
        print("Sampling additional data...")
        # Flatten to (year, collection, count) for proportional allocation
        add_entries = []
        for year in target_years:
            for coll, count in year_additional[year].items():
                add_entries.append((year, coll, count))
        
        total_add_counted = sum(c for _, _, c in add_entries)
        
        for year, coll, count in tqdm(add_entries, desc="Additional"):
            year_coll_n = max(1, round(n_additional * count / total_add_counted))
            samples = sample_additional(year, coll, year_coll_n, rng)
            all_samples.extend(samples)
    
    # Phase 4: Shuffle and save
    start_year = min(target_years)
    end_year = max(target_years)
    output_path = SAMPLE_OUTPUT_DIR / f"training_samples_{start_year}_{end_year}.parquet"
    
    final_df = pd.DataFrame(all_samples)
    final_df = final_df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
    final_df.to_parquet(output_path, index=False)
    
    # Print source breakdown
    source_counts = final_df["source"].value_counts()
    print(f"\nSaved {len(final_df)} samples to {output_path.name}")
    print("Source breakdown:")
    for src, cnt in source_counts.items():
        print(f"  {src}: {cnt} ({100*cnt/len(final_df):.1f}%)")

In [ ]:
# Generate all 14 periods and sample each
all_periods = [range(1678, 1701)]
for start in range(1701, 2002, 25):
    end = min(start + 24, 2023)
    all_periods.append(range(start, end + 1))

# Run for all periods (or subset as needed)
for period_range in all_periods:
    print(f"\n{'='*60}")
    print(f"Processing Period: {min(period_range)}_{max(period_range)}")
    print(f"{'='*60}")
    collect_combined_samples(list(period_range))

In [2]:
all_periods

[range(1678, 1701),
 range(1701, 1726),
 range(1726, 1751),
 range(1751, 1776),
 range(1776, 1801),
 range(1801, 1826),
 range(1826, 1851),
 range(1851, 1876),
 range(1876, 1901),
 range(1901, 1926),
 range(1926, 1951),
 range(1951, 1976),
 range(1976, 2001),
 range(2001, 2024)]